# Phase 2 — Data Ingestion & Preprocessing
## Noida AQI Forecasting System

This notebook demonstrates and validates the complete preprocessing pipeline.
All production logic lives in `src/preprocessing/` — this notebook is for:
- Visual inspection of each pipeline stage
- Generating the data quality report
- Documenting preprocessing decisions for the research paper

**Do not add business logic here — use `src/preprocessing/` instead.**

In [ ]:
# ── Environment setup ────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Ensure project root is on the path (Colab-compatible)
PROJECT_ROOT = Path().resolve().parents[1]   # notebooks/02_preprocessing → project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.utils.config import load_config
from src.utils.logger import configure_logging
from src.utils.reproducibility import set_global_seed

# Load config
cfg = load_config(PROJECT_ROOT / 'configs' / 'default.yaml')
configure_logging(level='INFO')
set_global_seed(cfg.project.seed)

print(f'Station : {cfg.project.station_id}')
print(f'Seed    : {cfg.project.seed}')

## 1. Load Raw Data

In [ ]:
from src.preprocessing.loader import DataLoader

loader = DataLoader(cfg)
df_raw = loader.load()

print(f'Shape  : {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
print(f'Range  : {df_raw[cfg.data.datetime_col].min()} → {df_raw[cfg.data.datetime_col].max()}')
df_raw.head()

## 2. Schema Validation

In [ ]:
from src.preprocessing.validator import SchemaValidator

validator = SchemaValidator(cfg)
df_validated = validator.validate(df_raw)
report = validator.get_validation_report()
report.print_summary()

print('Renamed columns :', report.renamed_columns)
print('Missing optional:', report.missing_optional)

## 3. Data Cleaning

In [ ]:
from src.preprocessing.cleaner import DataCleaner

cleaner = DataCleaner(cfg)
df_cleaned = cleaner.fit_transform(df_validated)

print(f'Shape after cleaning : {df_cleaned.shape}')
print(f'Null count           : {df_cleaned.isna().sum().sum()}')
print(f'Dropped columns      : {cleaner.dropped_columns}')
print(f'Date range           : {df_cleaned.index.min()} → {df_cleaned.index.max()}')

## 4. Outlier Detection & Treatment

In [ ]:
from src.preprocessing.outlier_handler import OutlierHandler

outlier_handler = OutlierHandler(cfg)
df_clean = outlier_handler.fit_transform(df_cleaned)

summary = outlier_handler.get_outlier_summary()
print('Outlier treatment summary:')
print(summary.to_string(index=False))

## 5. Quality Report

In [ ]:
from src.preprocessing.quality_report import QualityReporter

reporter = QualityReporter(cfg)
quality_report = reporter.generate(
    df_raw=df_raw,
    df_clean=df_clean,
    validation_report=report,
    outlier_summary_df=summary,
)
quality_report.print_summary()

# Save JSON + HTML reports
saved_paths = reporter.save(quality_report)
print('Reports saved:', saved_paths)

## 6. Train / Val / Test Split Visualisation

In [ ]:
from src.preprocessing.pipeline import _time_series_split

train_df, val_df, test_df = _time_series_split(df_clean, cfg, cfg.project.station_id)

print(f'Train : {len(train_df):,} rows  {train_df.index.min()} → {train_df.index.max()}')
print(f'Val   : {len(val_df):,} rows  {val_df.index.min()} → {val_df.index.max()}')
print(f'Test  : {len(test_df):,} rows  {test_df.index.min()} → {test_df.index.max()}')

# Visualise split
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(train_df.index, train_df['aqi'], color='steelblue',  lw=0.7, label='Train')
ax.plot(val_df.index,   val_df['aqi'],   color='darkorange', lw=0.7, label='Val')
ax.plot(test_df.index,  test_df['aqi'],  color='crimson',    lw=0.7, label='Test')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.set_title('AQI — Train / Val / Test Split (chronological, no leakage)')
ax.set_ylabel('AQI')
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'outputs/figures/split_visualisation.png', dpi=150)
plt.show()

## 7. Full Pipeline (one-liner)
The cell above ran each step manually for inspection.  In production, use the single pipeline call:

In [ ]:
from src.preprocessing.pipeline import run_preprocessing_pipeline

result = run_preprocessing_pipeline(cfg, save_outputs=True, print_report=True)

print(f'\nResult summary:')
print(f'  station        : {result.station_id}')
print(f'  clean shape    : {result.clean.shape}')
print(f'  train rows     : {len(result.train):,}')
print(f'  val rows       : {len(result.val):,}')
print(f'  test rows      : {len(result.test):,}')
print(f'  elapsed        : {result.elapsed_seconds:.2f}s')